# ArduMedics Notebook 03: YOLOv8m-Pose Fall Detection Training

**Project**: ArduMedics - AI-Powered Healthcare Robot  
**Component**: Feature 4 - Camera-Based Fall Detection  
**Model**: YOLOv8m-Pose (Medium - Upper-Bound Accuracy & Teacher Model)  
**Author**: ArduMedics AI Team  
**Notebook**: 03 / 06  
**Previous Notebook**: 02_YOLOv8s_Pose_Fall_Detection_Training  
**Next Notebook**: 04_YOLOv8n_Pose_Knowledge_Distillation  

---

## Objective
Train the **YOLOv8m-Pose (Medium)** model on ALL available fall detection + pose estimation
datasets. This is the **largest model** in our training pipeline and serves three critical roles:

1. **Ensemble Creation** (Notebook 06) - Combined with nano/small for boosted accuracy
2. **Teacher Model** for Knowledge Distillation (Notebook 04) - Transfers knowledge to nano
3. **Upper-Bound Accuracy Reference** for the paper - Demonstrates maximum achievable accuracy

> **Note**: This model is **too large for real-time Pi 5 deployment** (~26.4M params).
> Its value is in accuracy, not edge inference speed.

## Key Differences from Notebooks 01-02
| Parameter | NB01 (Nano) | NB02 (Small) | **NB03 (Medium)** |
|-----------|-------------|--------------|--------------------|
| Model | YOLOv8n-pose | YOLOv8s-pose | **YOLOv8m-pose** |
| Parameters | 3.2M | 11.2M | **26.4M** |
| Epochs | 150 | 120 | **80** |
| Batch Size | 16 | 8 | **4** |
| Learning Rate | 0.001 | 0.001 | **0.0005** |
| Primary Export | NCNN | NCNN | **ONNX** |

## Training Strategy
- **Model Size**: Medium (26.4M params) - Highest accuracy, not for edge
- **Epochs**: 80 (reduced due to much larger model + 12hr Kaggle limit)
- **Optimizer**: AdamW with cosine annealing LR
- **Batch Size**: 4 (GPU memory constraint for medium model)
- **Learning Rate**: 0.0005 (lower for larger model stability)
- **Image Size**: 640x640
- **Augmentation**: Mosaic + MixUp + reduced copy-paste/rotation for stability
- **Export Formats**: ONNX (primary), TFLite & NCNN optional

## Datasets Used
### Roboflow Pose Datasets (Keypoint-annotated for YOLOv8-Pose training):
1. **Falling Pose Estimation** (635 images) - https://universe.roboflow.com/humna-pose-data/falling-pose-estimation
2. **YOLOv8-Pose Fall Detection** (474 images, 2-class: fall/not-fallen) - https://universe.roboflow.com/yolo-xvnzo/yolov8-pose-utovc
> **NOTE**: nafzzan/falling-pose-estimation-0xme8 was **removed** — verified pixel-identical duplicate of Dataset 1 (same 635 images).
### Kaggle Fall Detection Datasets (For additional frame extraction + evaluation):
4. **UR Fall Detection Dataset** - https://www.kaggle.com/datasets/shahliza27/ur-fall-detection-dataset
5. **Fall Detection Dataset (Images)** - https://www.kaggle.com/datasets/uttejkumarkandagatla/fall-detection-dataset
6. **Le2i Fall Dataset** - https://www.kaggle.com/datasets/tuyenldvn/falldataset-imvia
7. **Multiple Cameras Fall Dataset** - https://www.kaggle.com/datasets/soumicksarker/multiple-cameras-fall-dataset
8. **Fall Video Dataset (Combined)** - https://www.kaggle.com/datasets/payutch/fall-video-dataset

**Estimated Training Time**: ~10-11 hours on T4 GPU (approaches 12-hour Kaggle limit)


---
## Step 1: Environment Setup

Install all required packages and configure the environment:
- `ultralytics` for YOLOv8-Pose training and export
- `roboflow` for downloading pose-annotated datasets
- `opencv-python-headless` for video frame extraction (headless for Kaggle)

> **IMPORTANT**: The medium model requires more GPU memory. Ensure T4 GPU is enabled.

In [1]:
# ============================================================
# OUTPUT MANAGEMENT UTILITIES (ArduMedics Standard)
# Suppresses noisy output while keeping important logs
# ============================================================

import os, sys, warnings, contextlib, io

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)

os.environ['OPENCV_LOG_LEVEL'] = 'ERROR'
os.environ['OPENCV_VIDEOIO_DEBUG'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['FLAGS_logtostderr'] = '0'
os.environ['GLOG_minloglevel'] = '3'
os.environ['YOLO_AUTODOWNLOAD'] = '1'

class suppress_output:
    def __enter__(self):
        self._orig = sys.stdout
        sys.stdout = io.StringIO()
        return self
    def __exit__(self, *a):
        sys.stdout = self._orig
        return False

_real_stdout = sys.stdout
def important_print(msg, end='\n'):
    _real_stdout.write(str(msg) + end)
    _real_stdout.flush()

print('[ArduMedics] Output management loaded')


[ArduMedics] Output management loaded


In [2]:
# ============================================================
# STEP 1: Environment Setup
# Notebook: 03 | Step: 1 of 8
# After this: Download Roboflow pose datasets
# ============================================================

!pip install -qq ultralytics roboflow opencv-python-headless

import os
import shutil
import yaml
import json
import cv2
cv2.setLogLevel(0)
import glob
import random
import numpy as np
from pathlib import Path
from datetime import datetime

# Verify ultralytics version (8.1+ required for proper pose export)
from ultralytics import YOLO
import ultralytics
print(f"Ultralytics version: {ultralytics.__version__}")

# Check GPU availability and memory
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("WARNING: No GPU detected! Medium model requires GPU.")

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

print(f"\nRandom seed set to: {SEED}")
print("\u2713 Step 1 complete: Environment ready for YOLOv8m-Pose training")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.9/207.9 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 89.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.2.0 which is incompatible.
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo se

---
## Step 2: Load Roboflow Pose Datasets

These datasets have **keypoint annotations** in YOLOv8-Pose format, which is exactly
what we need for training the pose estimation model. We load both Roboflow datasets
and merge them into a single unified training set.

> **ROBUST LOADING**: The code **recursively scans ALL of `/kaggle/input/`** for
> `data.yaml` files, then **copies datasets to `/kaggle/working/`** (writable directory).
> This handles any dataset structure — single ZIP, multiple uploads, or nested folders.
> **Never uses `/kaggle/input/` as a working directory** (read-only on Kaggle).

**Class Mapping Strategy**:
- Dataset 1 (humna): class 0='person' → remapped to class 0='fall'
- Dataset 2 (yolo-xvnzo): class 0='fall', class 1='not-fallen' → kept as-is
- Unified: **nc=2**, names=['fall', 'not-fallen']

**Fallback**: If no pre-uploaded dataset is found, the code downloads from Roboflow SDK
(requires a free Roboflow API key from https://app.roboflow.com/settings).

In [3]:
# ============================================================
# STEP 2: Download/Load Roboflow Pose Datasets
# Notebook: 03 | Step: 2 of 8
# After this: Download Kaggle fall datasets for frame extraction
# ============================================================
#
# ROBUST DATASET LOADING: 4-tier approach
#   1. Recursively scan ALL of /kaggle/input/ for data.yaml
#   2. Download from Kaggle dataset (nishatfifa/ardumedics-roboflow-pose-datasets)
#   3. Fallback: Roboflow SDK (requires API key)

DATASET_DIR = "/kaggle/working/datasets"
os.makedirs(DATASET_DIR, exist_ok=True)

roboflow_datasets_loaded = False

DATASET1_PATTERNS = ['falling_pose_estimation', 'falling-pose-estimation',
                     'Falling pose estimation', 'humna']
DATASET2_PATTERNS = ['yolov8_pose_fall', 'yolov8-pose',
                     'yolov8-pose.v1i', 'yolov8-pose.v1xme8', 'yolo-xvnzo']

found_ds1 = None
found_ds2 = None

if os.path.exists('/kaggle/input'):
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'data.yaml' in files:
            root_lower = root.lower()
            if any(p.lower() in root_lower for p in DATASET1_PATTERNS):
                found_ds1 = root
                print(f"  Found Dataset 1 at: {root}")
            elif any(p.lower() in root_lower for p in DATASET2_PATTERNS):
                found_ds2 = root
                print(f"  Found Dataset 2 at: {root}")
            else:
                if found_ds1 is None:
                    found_ds1 = root
                    print(f"  Found unknown dataset, assigning as Dataset 1: {root}")
                elif found_ds2 is None:
                    found_ds2 = root
                    print(f"  Found unknown dataset, assigning as Dataset 2: {root}")

if found_ds1:
    dst = f"{DATASET_DIR}/falling_pose_1"
    if not os.path.exists(dst):
        shutil.copytree(found_ds1, dst)
    img_count = 0
    for split in ['train', 'valid', 'val', 'test']:
        img_dir = os.path.join(found_ds1, split, 'images')
        if os.path.exists(img_dir):
            img_count += len([f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.png', '.jpeg'))])
    print(f"  Dataset 1 → falling_pose_1/ ({img_count} images)")

if found_ds2:
    dst = f"{DATASET_DIR}/falling_pose_2"
    if not os.path.exists(dst):
        shutil.copytree(found_ds2, dst)
    img_count = 0
    for split in ['train', 'valid', 'val', 'test']:
        img_dir = os.path.join(found_ds2, split, 'images')
        if os.path.exists(img_dir):
            img_count += len([f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.png', '.jpeg'))])
    print(f"  Dataset 2 → falling_pose_2/ ({img_count} images)")

if found_ds1 or found_ds2:
    roboflow_datasets_loaded = True
    print("  Datasets copied to writable directory!")

# ── Tier 3: Try downloading from Kaggle dataset ──
if not roboflow_datasets_loaded:
    print("[Tier 3] Trying Kaggle dataset download...")
    try:
        kaggle_ds_dir = f"{DATASET_DIR}/ardumedics_roboflow_temp"
        os.makedirs(kaggle_ds_dir, exist_ok=True)
        !kaggle datasets download -d nishatfifa/ardumedics-roboflow-pose-datasets -p {kaggle_ds_dir} --unzip
        
        # Scan the downloaded directory for data.yaml
        for root, dirs, files in os.walk(kaggle_ds_dir):
            if 'data.yaml' in files:
                root_lower = root.lower()
                if any(p.lower() in root_lower for p in DATASET1_PATTERNS) and found_ds1 is None:
                    found_ds1 = root
                    print(f"  Found Dataset 1 at: {root}")
                elif any(p.lower() in root_lower for p in DATASET2_PATTERNS) and found_ds2 is None:
                    found_ds2 = root
                    print(f"  Found Dataset 2 at: {root}")
                elif found_ds1 is None:
                    found_ds1 = root
                    print(f"  Found unknown dataset, assigning as Dataset 1: {root}")
                elif found_ds2 is None:
                    found_ds2 = root
                    print(f"  Found unknown dataset, assigning as Dataset 2: {root}")
        
        # Copy found datasets
        if found_ds1:
            dst = f"{DATASET_DIR}/falling_pose_1"
            if not os.path.exists(dst):
                shutil.copytree(found_ds1, dst)
            print(f"  Dataset 1 → falling_pose_1/")
        
        if found_ds2:
            dst = f"{DATASET_DIR}/falling_pose_2"
            if not os.path.exists(dst):
                shutil.copytree(found_ds2, dst)
            print(f"  Dataset 2 → falling_pose_2/")
        
        if found_ds1 or found_ds2:
            roboflow_datasets_loaded = True
            print("Datasets downloaded from Kaggle and copied!")
            shutil.rmtree(kaggle_ds_dir, ignore_errors=True)
        else:
            print("  Kaggle download succeeded but no data.yaml found inside.")
            shutil.rmtree(kaggle_ds_dir, ignore_errors=True)
    except Exception as e:
        print(f"  Kaggle dataset download failed: {e}")
        temp_dir = f"{DATASET_DIR}/ardumedics_roboflow_temp"
        if os.path.exists(temp_dir):
            shutil.rmtree(temp_dir, ignore_errors=True)

# ── Tier 4: Fallback — Roboflow SDK download ──
if not roboflow_datasets_loaded:
    print("[Fallback] Downloading from Roboflow SDK...")
    ROBOFLOW_API_KEY = os.environ.get("ROBOFLOW_API_KEY", "YOUR_ROBOFLOW_API_KEY_HERE")
    
    if ROBOFLOW_API_KEY == "YOUR_ROBOFLOW_API_KEY_HERE":
        print("=" * 60)
        print("ERROR: No datasets found and Roboflow API key not set!")
        print("=" * 60)
        print("To fix this, do ONE of:")
        print("  1. Add 'nishatfifa/ardumedics-roboflow-pose-datasets' as")
        print("     a Kaggle dataset input to this notebook")
        print("  2. Set ROBOFLOW_API_KEY environment variable")
        print("  3. Upload a ZIP containing falling_pose_estimation/ and")
        print("     yolov8_pose_fall/ to Kaggle and add as input")
        print("=" * 60)
        raise ValueError("No datasets available and Roboflow API key not configured")
    
    from roboflow import Roboflow
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    
    print("Downloading Falling Pose Estimation (635 images)...")
    with suppress_output():
        p1 = rf.workspace("humna-pose-data").project("falling-pose-estimation")
        dataset1 = p1.version(2).download("yolov8", location=f"{DATASET_DIR}/falling_pose_1")
    print(f"  Done!")
    
    print("Downloading YOLOv8-Pose Fall Detection (474 images)...")
    with suppress_output():
        p2 = rf.workspace("yolo-xvnzo").project("yolov8-pose-utovc")
        dataset2 = p2.version(3).download("yolov8", location=f"{DATASET_DIR}/falling_pose_2")
    print(f"  Done!")

print("\n✓ Step 2 complete: Pose datasets ready")

  Found Dataset 1 at: /kaggle/input/datasets/nishatfifa/ardumedics-roboflow-pose-datasets/ardumedics-roboflow-pose-datasets/falling_pose_estimation
  Found Dataset 2 at: /kaggle/input/datasets/nishatfifa/ardumedics-roboflow-pose-datasets/ardumedics-roboflow-pose-datasets/yolov8_pose_fall
  Dataset 1 → falling_pose_1/ (635 images)
  Dataset 2 → falling_pose_2/ (474 images)
  Datasets copied to writable directory!

✓ Step 2 complete: Pose datasets ready


---
## Step 3: Download Kaggle Fall Datasets & Extract Frames

The Kaggle datasets contain fall/non-fall images and videos. We:
1. Add image-based datasets directly to training data (with auto-labeling using pre-trained YOLOv8-Pose)
2. Extract frames from video datasets for training data expansion

**Kaggle Datasets**: Add these to your Kaggle notebook before running:
- `shahliza27/ur-fall-detection-dataset`
- `uttejkumarkandagatla/fall-detection-dataset`
- `tuyenldvn/falldataset-imvia`
- `soumicksarker/multiple-cameras-fall-dataset`
- `payutch/fall-video-dataset`

In [4]:
# ============================================================
# STEP 3: Download Kaggle fall datasets & extract frames
# Notebook: 03 | Step: 3 of 8
# After this: Auto-label Kaggle frames with pre-trained pose model
# ============================================================

# Kaggle datasets are auto-mounted at /kaggle/input/
# Check what's available
kaggle_input = "/kaggle/input"
print("Available Kaggle datasets:")
for d in sorted(os.listdir(kaggle_input)):
    print(f"  - {d}")

# ---- Frame extraction function for video datasets ----
def extract_frames_from_videos(video_dir, output_dir, sample_rate=5):
    """
    Extract frames from video files at given sample rate.
    
    Args:
        video_dir: Directory containing video files
        output_dir: Directory to save extracted frames
        sample_rate: Extract 1 frame every N frames (default: every 5th frame)
    
    Returns:
        Number of frames extracted
    """
    os.makedirs(output_dir, exist_ok=True)
    count = 0
    
    video_extensions = ['.avi', '.mp4', '.mov', '.mkv', '.wmv']
    video_files = []
    for ext in video_extensions:
        video_files.extend(glob.glob(os.path.join(video_dir, '**', f'*{ext}'), recursive=True))
    
    for vf in video_files:
        cap = cv2.VideoCapture(vf)
        frame_idx = 0
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            if frame_idx % sample_rate == 0:
                frame_path = os.path.join(output_dir, f"frame_{count:06d}.jpg")
                cv2.imwrite(frame_path, frame)
                count += 1
            frame_idx += 1
        cap.release()
    
    return count

# ---- Process UR Fall Detection Dataset ----
# URL: https://www.kaggle.com/datasets/shahliza27/ur-fall-detection-dataset
ur_fall_path = None
for d in os.listdir(kaggle_input):
    if 'ur-fall' in d.lower():
        ur_fall_path = os.path.join(kaggle_input, d)
        break

if ur_fall_path:
    print(f"\nProcessing UR Fall Dataset: {ur_fall_path}")
    ur_frames_dir = f"{DATASET_DIR}/ur_fall_frames"
    n_frames = extract_frames_from_videos(ur_fall_path, ur_frames_dir, sample_rate=3)
    print(f"  Extracted {n_frames} frames")

# ---- Process Fall Detection Images Dataset ----
# URL: https://www.kaggle.com/datasets/uttejkumarkandagatla/fall-detection-dataset
fall_img_path = None
for d in os.listdir(kaggle_input):
    if 'fall-detection-dataset' in d.lower():
        fall_img_path = os.path.join(kaggle_input, d)
        break

if fall_img_path:
    print(f"\nProcessing Fall Detection Images: {fall_img_path}")
    fall_img_dest = f"{DATASET_DIR}/fall_detection_images"
    os.makedirs(fall_img_dest, exist_ok=True)
    img_count = 0
    for ext in ['*.jpg', '*.png', '*.jpeg']:
        for f in glob.glob(os.path.join(fall_img_path, '**', ext), recursive=True):
            shutil.copy2(f, os.path.join(fall_img_dest, f"img_{img_count:06d}.jpg"))
            img_count += 1
    print(f"  Copied {img_count} images")

# ---- Process Le2i Fall Dataset ----
# URL: https://www.kaggle.com/datasets/tuyenldvn/falldataset-imvia
le2i_path = None
for d in os.listdir(kaggle_input):
    if 'falldataset' in d.lower() or 'le2i' in d.lower() or 'imvia' in d.lower():
        le2i_path = os.path.join(kaggle_input, d)
        break

if le2i_path:
    print(f"\nProcessing Le2i Fall Dataset: {le2i_path}")
    le2i_frames_dir = f"{DATASET_DIR}/le2i_frames"
    n_frames = extract_frames_from_videos(le2i_path, le2i_frames_dir, sample_rate=5)
    print(f"  Extracted {n_frames} frames")

# ---- Process Multiple Cameras Fall Dataset ----
# URL: https://www.kaggle.com/datasets/soumicksarker/multiple-cameras-fall-dataset
multi_cam_path = None
for d in os.listdir(kaggle_input):
    if 'multiple-cameras' in d.lower():
        multi_cam_path = os.path.join(kaggle_input, d)
        break

if multi_cam_path:
    print(f"\nProcessing Multiple Cameras Fall Dataset: {multi_cam_path}")
    multi_frames_dir = f"{DATASET_DIR}/multi_cam_frames"
    n_frames = extract_frames_from_videos(multi_cam_path, multi_frames_dir, sample_rate=5)
    print(f"  Extracted {n_frames} frames")

# ---- Process Fall Video Dataset (Combined) ----
# URL: https://www.kaggle.com/datasets/payutch/fall-video-dataset
fall_vid_path = None
for d in os.listdir(kaggle_input):
    if 'fall-video' in d.lower():
        fall_vid_path = os.path.join(kaggle_input, d)
        break

if fall_vid_path:
    print(f"\nProcessing Fall Video Dataset: {fall_vid_path}")
    fallvid_frames_dir = f"{DATASET_DIR}/fall_video_frames"
    n_frames = extract_frames_from_videos(fall_vid_path, fallvid_frames_dir, sample_rate=5)
    print(f"  Extracted {n_frames} frames")

# Disk status check after processing all datasets
disk_stat = shutil.disk_usage("/kaggle/working")
print(f"\nDisk usage: {disk_stat.used / 1e9:.1f} GB / {disk_stat.total / 1e9:.1f} GB ({disk_stat.free / 1e9:.1f} GB free)")
if disk_stat.free < 5e9:
    print("WARNING: Low disk space! Cleaning up temporary files...")
    for zf in Path("/kaggle/working").rglob("*.zip"):
        zf.unlink()
        print(f"  Deleted: {zf}")

print("\n\u2713 Step 3 complete: Kaggle datasets processed")

Available Kaggle datasets:
  - datasets

Disk usage: 0.1 GB / 21.0 GB (20.9 GB free)

✓ Step 3 complete: Kaggle datasets processed


---
## Step 4: Auto-Label Kaggle Frames with Pre-trained YOLOv8-Pose

The Kaggle image/video datasets don't have keypoint annotations. We use the pre-trained
COCO YOLOv8n-Pose model to **auto-label** these images, then add them to our training set.
This is a form of **pseudo-labeling** that significantly expands our training data.

> Using the nano model for auto-labeling (not medium) to keep this step fast.

In [5]:
# ============================================================
# STEP 4: Auto-label Kaggle frames using pre-trained YOLOv8-Pose
# Notebook: 03 | Step: 4 of 8
# After this: Merge all datasets into unified training set
# ============================================================

# Load pre-trained COCO pose model for auto-labeling (use nano for speed)
auto_label_model = YOLO('yolov8n-pose.pt')

def auto_label_images(images_dir, output_labels_dir, conf_threshold=0.5):
    """
    Run pre-trained YOLOv8-Pose on images and save keypoint predictions as labels.
    Only saves labels where a person is detected with confidence > threshold.
    
    Args:
        images_dir: Directory containing images to label
        output_labels_dir: Directory to save YOLOv8-Pose format labels
        conf_threshold: Minimum confidence for keeping predictions
    
    Returns:
        Number of successfully labeled images
    """
    os.makedirs(output_labels_dir, exist_ok=True)
    labeled_count = 0
    
    image_files = []
    for ext in ['*.jpg', '*.png', '*.jpeg']:
        image_files.extend(glob.glob(os.path.join(images_dir, ext)))
    
    print(f"  Auto-labeling {len(image_files)} images from {images_dir}...")
    
    for img_path in image_files:
        results = auto_label_model(img_path, verbose=False)
        
        if len(results) > 0 and results[0].keypoints is not None:
            result = results[0]
            boxes = result.boxes
            keypoints = result.keypoints
            
            if boxes is not None and len(boxes) > 0:
                mask = boxes.conf >= conf_threshold
                if mask.any():
                    img_name = Path(img_path).stem
                    label_path = os.path.join(output_labels_dir, f"{img_name}.txt")
                    
                    with open(label_path, 'w') as f:
                        for i in range(len(boxes)):
                            if mask[i]:
                                cls = int(boxes.cls[i])
                                box = boxes.xywhn[i].cpu().numpy()
                                kpts = keypoints.xyn[i].cpu().numpy().flatten()
                                
                                line = f"{cls} {box[0]:.6f} {box[1]:.6f} {box[2]:.6f} {box[3]:.6f}"
                                for k in range(0, len(kpts), 3):
                                    line += f" {kpts[k]:.6f} {kpts[k+1]:.6f} {int(kpts[k+2])}"
                                f.write(line + "\n")
                    labeled_count += 1
    
    return labeled_count

# Auto-label each extracted frame set
labeled_dirs = {}
for frame_dir_name in ['ur_fall_frames', 'le2i_frames', 'multi_cam_frames', 
                        'fall_video_frames', 'fall_detection_images']:
    frame_dir = os.path.join(DATASET_DIR, frame_dir_name)
    if os.path.exists(frame_dir):
        label_dir = os.path.join(DATASET_DIR, f"{frame_dir_name}_labels")
        n_labeled = auto_label_images(frame_dir, label_dir, conf_threshold=0.5)
        labeled_dirs[frame_dir_name] = {
            'images': frame_dir,
            'labels': label_dir,
            'count': n_labeled
        }
        print(f"  Labeled {n_labeled} images from {frame_dir_name}")

print("\n\u2713 Step 4 complete: Auto-labeling finished")


✓ Step 4 complete: Auto-labeling finished


---
## Step 5: Merge All Datasets into Unified YOLOv8-Pose Format

Combine the 2 Roboflow pose datasets + auto-labeled Kaggle frames into a single
unified dataset with proper train/val/test split.

> Same dataset as Notebooks 01-02, ensuring fair comparison across model sizes.

In [6]:
# ============================================================
# STEP 5: Merge all datasets into unified YOLOv8-Pose format
# Notebook: 03 | Step: 5 of 8
# After this: Verify dataset integrity and create data.yaml
# ============================================================

UNIFIED_DIR = f"{DATASET_DIR}/ardumedics_unified_pose"

def remap_label_file(src_label, dst_label, class_map):
    """Remap class IDs in a YOLOv8 label file."""
    with open(src_label, 'r') as f:
        lines = f.readlines()
    remapped = []
    for line in lines:
        parts = line.strip().split()
        if len(parts) >= 5:
            old_cls = int(parts[0])
            if old_cls in class_map:
                parts[0] = str(class_map[old_cls])
        remapped.append(' '.join(parts))
    with open(dst_label, 'w') as f:
        f.write('\n'.join(remapped) + '\n')

def merge_datasets(roboflow_datasets, auto_labeled_dirs, output_dir, train_ratio=0.8, val_ratio=0.15):
    """
    Merge multiple YOLOv8-Pose datasets into one unified dataset.
    
    Args:
        roboflow_datasets: List of (path, class_map) tuples for Roboflow datasets
        auto_labeled_dirs: Dict of auto-labeled image/label directories
        output_dir: Output directory for unified dataset
        train_ratio: Fraction for training set
        val_ratio: Fraction for validation set (rest goes to test)
    
    Returns:
        Path to unified dataset
    """
    for split in ['train', 'val', 'test']:
        os.makedirs(f"{output_dir}/{split}/images", exist_ok=True)
        os.makedirs(f"{output_dir}/{split}/labels", exist_ok=True)
    
    all_pairs = []
    
    for ds_path, class_map in roboflow_datasets:
        ds_path = str(ds_path)
        for split in ['train', 'valid', 'val', 'test']:
            img_dir = os.path.join(ds_path, split, 'images')
            lbl_dir = os.path.join(ds_path, split, 'labels')
            if os.path.exists(img_dir) and os.path.exists(lbl_dir):
                for img_file in os.listdir(img_dir):
                    if img_file.lower().endswith(('.jpg', '.png', '.jpeg')):
                        lbl_file = img_file.rsplit('.', 1)[0] + '.txt'
                        lbl_path = os.path.join(lbl_dir, lbl_file)
                        if os.path.exists(lbl_path):
                            if os.path.getsize(lbl_path) > 0:
                                all_pairs.append((
                                    os.path.join(img_dir, img_file),
                                    lbl_path,
                                    class_map
                                ))
    
    for name, info in auto_labeled_dirs.items():
        img_dir = info['images']
        lbl_dir = info['labels']
        if os.path.exists(img_dir) and os.path.exists(lbl_dir):
            for img_file in os.listdir(img_dir):
                if img_file.lower().endswith(('.jpg', '.png', '.jpeg')):
                    lbl_file = img_file.rsplit('.', 1)[0] + '.txt'
                    lbl_path = os.path.join(lbl_dir, lbl_file)
                    if os.path.exists(lbl_path) and os.path.getsize(lbl_path) > 0:
                        all_pairs.append((
                            os.path.join(img_dir, img_file),
                            lbl_path,
                            {}  # no remap for auto-labeled
                        ))
    
    random.shuffle(all_pairs)
    n_total = len(all_pairs)
    n_train = int(n_total * train_ratio)
    n_val = int(n_total * val_ratio)
    
    train_pairs = all_pairs[:n_train]
    val_pairs = all_pairs[n_train:n_train + n_val]
    test_pairs = all_pairs[n_train + n_val:]
    
    def copy_pairs(pairs, split_name):
        for i, (img_path, lbl_path, class_map) in enumerate(pairs):
            img_ext = Path(img_path).suffix
            dst_img = f"{output_dir}/{split_name}/images/{split_name}_{i:06d}{img_ext}"
            dst_lbl = f"{output_dir}/{split_name}/labels/{split_name}_{i:06d}.txt"
            shutil.copy2(img_path, dst_img)
            if class_map:
                remap_label_file(lbl_path, dst_lbl, class_map)
            else:
                shutil.copy2(lbl_path, dst_lbl)
    
    copy_pairs(train_pairs, 'train')
    copy_pairs(val_pairs, 'val')
    copy_pairs(test_pairs, 'test')
    
    print(f"Unified dataset created at: {output_dir}")
    print(f"  Train: {len(train_pairs)} images")
    print(f"  Val:   {len(val_pairs)} images")
    print(f"  Test:  {len(test_pairs)} images")
    print(f"  Total: {n_total} images")
    
    return output_dir

roboflow_datasets_config = [
    {'path': f'{DATASET_DIR}/falling_pose_1', 'class_map': {0: 0}},  # person → fall
    {'path': f'{DATASET_DIR}/falling_pose_2', 'class_map': {0: 0, 1: 1}},  # fall→fall, not-fallen→not-fallen
]
roboflow_paths = [(cfg['path'], cfg['class_map']) for cfg in roboflow_datasets_config if os.path.exists(cfg['path'])]

unified_path = merge_datasets(roboflow_paths, labeled_dirs, UNIFIED_DIR)

print("\n\u2713 Step 5 complete: Datasets merged")

Unified dataset created at: /kaggle/working/datasets/ardumedics_unified_pose
  Train: 818 images
  Val:   153 images
  Test:  52 images
  Total: 1023 images

✓ Step 5 complete: Datasets merged


In [7]:
# ============================================================
# STEP 5b: Free disk space by removing intermediate files
# ============================================================

cleanup_dirs = [
    'ur_fall_frames', 'ur_fall_frames_labels',
    'le2i_frames', 'le2i_frames_labels',
    'multi_cam_frames', 'multi_cam_frames_labels',
    'fall_video_frames', 'fall_video_frames_labels',
    'fall_detection_images', 'fall_detection_images_labels',
    'falling_pose_1', 'falling_pose_2',
    'ardumedics_roboflow_temp',  # Clean up temp from Tier 3 download
]

freed_mb = 0
for dir_name in cleanup_dirs:
    dir_path = os.path.join(DATASET_DIR, dir_name)
    if os.path.exists(dir_path):
        total_size = 0
        for root, dirs, files in os.walk(dir_path):
            for f in files:
                fp = os.path.join(root, f)
                if os.path.exists(fp):
                    total_size += os.path.getsize(fp)
        freed_mb += total_size / (1024 * 1024)
        shutil.rmtree(dir_path, ignore_errors=True)
        print(f'  Deleted {dir_name}/ ({total_size/(1024*1024):.1f} MB)')

print(f'\nTotal freed: {freed_mb:.1f} MB')
print('✓ Step 5b complete: Disk space freed for training')

  Deleted falling_pose_1/ (36.4 MB)
  Deleted falling_pose_2/ (15.2 MB)

Total freed: 51.7 MB
✓ Step 5b complete: Disk space freed for training


---
## Step 5b: Free Disk Space (Kaggle 20GB Limit)

Delete intermediate files to free disk space before training.
Kaggle `/kaggle/working/` is limited to **20GB**.

---
## Step 6: Create data.yaml and Verify Dataset

Create the YOLOv8 data configuration file and verify that all labels are valid
YOLOv8-Pose format before training.

In [8]:
# ============================================================
# STEP 6: Create data.yaml and verify dataset integrity
# Notebook: 03 | Step: 6 of 8
# After this: Train YOLOv8m-Pose model
# ============================================================

def verify_labels(dataset_dir, split='train', max_check=100):
    """Verify label files have correct format for YOLOv8-Pose."""
    label_dir = os.path.join(dataset_dir, split, 'labels')
    if not os.path.exists(label_dir):
        print(f"  No {split} labels directory found!")
        return False
    
    label_files = glob.glob(os.path.join(label_dir, '*.txt'))
    errors = 0
    
    for lf in label_files[:max_check]:
        with open(lf, 'r') as f:
            for line_idx, line in enumerate(f):
                parts = line.strip().split()
                if len(parts) < 5:
                    errors += 1
                    continue
                try:
                    cls = int(parts[0])
                except ValueError:
                    errors += 1
                    continue
    
    total = min(len(label_files), max_check)
    print(f"  {split}: Checked {total} label files, {errors} errors")
    return errors == 0

for split in ['train', 'val', 'test']:
    verify_labels(UNIFIED_DIR, split)

data_yaml = {
    'path': UNIFIED_DIR,
    'train': 'train/images',
    'val': 'val/images',
    'test': 'test/images',
    'names': {0: 'fall', 1: 'not-fallen'},
    'kpt_shape': [17, 3],
    'flip_idx': [0, 2, 1, 4, 3, 6, 5, 8, 7, 10, 9, 12, 11, 14, 13, 16, 15]
}

yaml_path = f"{DATASET_DIR}/ardumedics_pose.yaml"
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print(f"\ndata.yaml created at: {yaml_path}")
print(f"  Dataset path: {UNIFIED_DIR}")
print(f"  Keypoint shape: 17 keypoints x 3 (x, y, visibility)")

total_images = 0
for split in ['train', 'val', 'test']:
    img_dir = os.path.join(UNIFIED_DIR, split, 'images')
    if os.path.exists(img_dir):
        n = len(os.listdir(img_dir))
        total_images += n
        print(f"  {split}: {n} images")
print(f"  TOTAL: {total_images} images")

print("\n\u2713 Step 6 complete: Dataset verified and data.yaml created")

  train: Checked 100 label files, 0 errors
  val: Checked 100 label files, 0 errors
  test: Checked 52 label files, 0 errors

data.yaml created at: /kaggle/working/datasets/ardumedics_pose.yaml
  Dataset path: /kaggle/working/datasets/ardumedics_unified_pose
  Keypoint shape: 17 keypoints x 3 (x, y, visibility)
  train: 818 images
  val: 153 images
  test: 52 images
  TOTAL: 1023 images

✓ Step 6 complete: Dataset verified and data.yaml created


---
## Step 7: Train YOLOv8m-Pose Model

This is the main training step for the **Medium** pose model. Key considerations:

- **80 epochs** (reduced from 150/120 due to much larger model + time budget)
- **Batch size 4** (T4 16GB VRAM constraint for 26.4M param model)
- **Lower learning rate (0.0005)** for stability with larger model
- **Reduced rotation (10 deg)** to prevent unstable training on larger model
- **Slightly reduced copy_paste (0.2)** for training stability
- **AdamW optimizer** with cosine annealing for smooth convergence

> **TIME WARNING**: ~10-11 hours on T4. Monitor GPU temperature and memory usage.

In [9]:
# ============================================================
# STEP 7: Train YOLOv8m-Pose model
# Notebook: 03 | Step: 7 of 8
# After this: Evaluate and export the trained model
# ============================================================

# Initialize model from pre-trained COCO weights
# YOLOv8m-pose: 26.4M parameters - largest model in our pipeline
model = YOLO('yolov8m-pose.pt')

# Print model info
print("Model Summary:")
model.info(verbose=True)

# Training configuration - OPTIMIZED for YOLOv8m-Pose
# Key differences from NB01/NB02:
#   - Fewer epochs (80 vs 150/120) due to model size + time budget
#   - Batch size 4 (memory constraint for medium model on T4)
#   - Lower learning rate (0.0005) for training stability
#   - Reduced rotation (10 deg) for larger model stability
#   - Slightly reduced copy_paste (0.2)
TRAIN_CONFIG = {
    'data': yaml_path,
    'epochs': 80,             # Reduced for medium model (time budget)
    'imgsz': 640,             # Standard YOLO resolution
    'batch': 8,  # Doubled for T4x2 (was 4 for single T4)               # Further reduced for medium model (T4 VRAM)
    'patience': 25,           # Early stopping patience
    'device': [0, 1],  # T4x2 Dual GPU (DDP)              # GPU
    'seed': SEED,
    'workers': 4,  # Kaggle has 4 CPU cores  # Kaggle has 4 CPU cores             # Kaggle T4 has 4 CPU cores
    'optimizer': 'AdamW',     # Better convergence for larger models
    'lr0': 0.0005,            # Lower LR for larger model stability
    'lrf': 0.01,              # Final LR factor (lr0 * lrf = 0.000005)
    'cos_lr': True,           # Cosine annealing schedule
    'warmup_epochs': 5,       # Gradual warmup

    # Augmentation settings - CONSERVATIVE for larger model
    'augment': True,
    'mosaic': 1.0,            # Mosaic augmentation probability
    'mixup': 0.15,            # MixUp augmentation (slightly increased for medium)
    'copy_paste': 0.2,        # Slightly reduced copy-paste for stability
    'degrees': 10.0,          # Less rotation for larger model stability
    'translate': 0.1,         # Translation augmentation
    'scale': 0.4,             # Scale augmentation
    'fliplr': 0.5,            # Horizontal flip probability
    'hsv_h': 0.015,           # HSV hue augmentation
    'hsv_s': 0.7,             # HSV saturation augmentation
    'hsv_v': 0.4,             # HSV value augmentation

    # Training settings
    'project': '/kaggle/working/runs',
    'name': 'ardumedics_medium_pose',
    'exist_ok': True,
    'pretrained': True,
    'verbose': True,

    # Logging & saving
    'val': True,              # Validate every epoch
    'plots': True,            # Generate training plots
    'save': True,             # Save checkpoints
    'save_period': 20,        # Save checkpoint every 20 epochs
}

print("\n" + "=" * 60)
print("YOLOv8m-Pose Training Configuration")
print("=" * 60)
print(f"  Model:        yolov8m-pose (Medium - 26.4M params)")
print(f"  Epochs:       {TRAIN_CONFIG['epochs']}")
print(f"  Batch size:   {TRAIN_CONFIG['batch']}")
print(f"  Optimizer:    {TRAIN_CONFIG['optimizer']}")
print(f"  Learning rate: {TRAIN_CONFIG['lr0']} (cosine annealing)")
print(f"  Final LR:     {TRAIN_CONFIG['lr0'] * TRAIN_CONFIG['lrf']}")
print(f"  Image size:   {TRAIN_CONFIG['imgsz']}")
print(f"  Patience:     {TRAIN_CONFIG['patience']}")
print(f"  Warmup:       {TRAIN_CONFIG['warmup_epochs']} epochs")
print(f"  Augmentations: mosaic={TRAIN_CONFIG['mosaic']}, mixup={TRAIN_CONFIG['mixup']}, "
      f"copy_paste={TRAIN_CONFIG['copy_paste']}")
print(f"  Rotation:     +/-{TRAIN_CONFIG['degrees']} degrees")
print(f"  Save period:  Every {TRAIN_CONFIG['save_period']} epochs")
print(f"  Estimated time: ~10-11 hours on T4")
print("=" * 60)
print()

# RUN TRAINING
results = model.train(**TRAIN_CONFIG)

print("\n\u2713 Step 7 complete: YOLOv8m-Pose training finished")

Model Summary:
YOLOv8m-pose summary: 184 layers, 26,464,462 parameters, 0 gradients, 81.4 GFLOPs

YOLOv8m-Pose Training Configuration
  Model:        yolov8m-pose (Medium - 26.4M params)
  Epochs:       80
  Batch size:   8
  Optimizer:    AdamW
  Learning rate: 0.0005 (cosine annealing)
  Final LR:     5e-06
  Image size:   640
  Patience:     25
  Warmup:       5 epochs
  Augmentations: mosaic=1.0, mixup=0.15, copy_paste=0.2
  Rotation:     +/-10.0 degrees
  Save period:  Every 20 epochs
  Estimated time: ~10-11 hours on T4

Ultralytics 8.4.51 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
                                                       CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.2, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/

---
## Step 8: Evaluate and Export

Evaluate the trained medium model on validation and test sets, then export to deployment formats.

**Export Strategy for Medium Model:**
- **ONNX** (primary) - Universal format, used for ensemble in Notebook 06
- **TFLite** (optional) - May be useful for Android deployment
- **NCNN** (optional) - Likely too slow on Pi 5 for real-time, but exported if needed

We also generate a **comparison table** with NB01 (Nano) and NB02 (Small) model sizes.

In [10]:
# ============================================================
# STEP 8: Evaluate and export the YOLOv8m-Pose model
# Notebook: 03 | Step: 8 of 8
# After this: Proceed to Notebook 04 (Knowledge Distillation)
# ============================================================

# Load best model from training
best_model_path = '/kaggle/working/runs/ardumedics_medium_pose/weights/best.pt'
best_model = YOLO(best_model_path)

# ---- Validation Set Evaluation ----
print("=" * 60)
print("VALIDATION SET EVALUATION - YOLOv8m-Pose (Medium)")
print("=" * 60)
val_results = best_model.val(data=yaml_path, split='val', verbose=True)

print(f"\nBox Metrics (Detection):")
print(f"  Precision: {val_results.box.mp:.4f}")
print(f"  Recall:    {val_results.box.mr:.4f}")
print(f"  mAP@50:    {val_results.box.map50:.4f}")
print(f"  mAP@50-95: {val_results.box.map:.4f}")

print(f"\nPose Metrics (Keypoints):")
print(f"  mAP@50 (Pose):    {val_results.pose.map50:.4f}")
print(f"  mAP@50-95 (Pose): {val_results.pose.map:.4f}")

# ---- Test Set Evaluation ----
print("\n" + "=" * 60)
print("TEST SET EVALUATION - YOLOv8m-Pose (Medium)")
print("=" * 60)
test_results = best_model.val(data=yaml_path, split='test', verbose=True)

print(f"\nBox Metrics (Detection):")
print(f"  Precision: {test_results.box.mp:.4f}")
print(f"  Recall:    {test_results.box.mr:.4f}")
print(f"  mAP@50:    {test_results.box.map50:.4f}")
print(f"  mAP@50-95: {test_results.box.map:.4f}")

print(f"\nPose Metrics (Keypoints):")
print(f"  mAP@50 (Pose):    {test_results.pose.map50:.4f}")
print(f"  mAP@50-95 (Pose): {test_results.pose.map:.4f}")

# ---- Save metrics to JSON for later comparison ----
metrics_03 = {
    'notebook': '03_YOLOv8m_Pose',
    'model': 'yolov8m-pose',
    'model_size': 'm',
    'best_mAP50': float(val_results.box.map50),
    'params_millions': 26.4,
    'timestamp': datetime.now().isoformat(),
    'training_config': {
        'epochs': 80,
        'batch': 4,
        'optimizer': 'AdamW',
        'lr0': 0.0005,
        'cos_lr': True,
        'imgsz': 640,
        'patience': 25,
    },
    'val': {
        'box_precision': float(val_results.box.mp),
        'box_recall': float(val_results.box.mr),
        'box_map50': float(val_results.box.map50),
        'box_map': float(val_results.box.map),
        'pose_map50': float(val_results.pose.map50),
        'pose_map': float(val_results.pose.map),
    },
    'test': {
        'box_precision': float(test_results.box.mp),
        'box_recall': float(test_results.box.mr),
        'box_map50': float(test_results.box.map50),
        'box_map': float(test_results.box.map),
        'pose_map50': float(test_results.pose.map50),
        'pose_map': float(test_results.pose.map),
    }
}

metrics_path = '/kaggle/working/metrics_notebook03_medium.json'
with open(metrics_path, 'w') as f:
    json.dump(metrics_03, f, indent=2)

print(f"\nMetrics saved to: {metrics_path}")

VALIDATION SET EVALUATION - YOLOv8m-Pose (Medium)
Ultralytics 8.4.51 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8m-pose summary (fused): 102 layers, 26,448,175 parameters, 0 gradients, 81.0 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1477.5±977.2 MB/s, size: 60.6 KB)
val: Scanning /kaggle/working/datasets/ardumedics_unified_pose/val/labels.cache... 153 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 153/153 40.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Pose(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 2.4it/s 4.1s
                   all        153        155      0.602       0.84      0.709      0.478      0.298      0.333      0.198     0.0505
                  fall        117        118      0.716      0.896      0.842      0.528      0.381      0.424      0.269     0.0885
            not-fallen         36         37      0.489      0.784      0.575      0.429      

In [11]:
# ============================================================
# STEP 8 (continued): Export model to deployment formats
# Notebook: 03 | Step: 8 of 8
# After this: Proceed to Notebook 04 (Knowledge Distillation)
# ============================================================

EXPORT_DIR = '/kaggle/working/exports'
os.makedirs(EXPORT_DIR, exist_ok=True)

# ---- Export 1: ONNX (PRIMARY for medium model) ----
print("Exporting to ONNX format (primary)...")
try:
    onnx_path = best_model.export(format='onnx', imgsz=640, simplify=True, dynamic=True)
    print(f"  ONNX exported to: {onnx_path}")
except Exception as e:
    print(f"  ONNX export failed: {e}")
    onnx_path = None

# ---- Export 2: TFLite (optional) ----
print("\nExporting to TFLite format (optional)...")
try:
    tflite_path = best_model.export(format='tflite', imgsz=640)
    print(f"  TFLite exported to: {tflite_path}")
except Exception as e:
    print(f"  TFLite export failed (optional, skipping): {e}")
    tflite_path = None

# ---- Export 3: NCNN (optional - likely too slow on Pi 5) ----
print("\nExporting to NCNN format (optional - may be slow on Pi 5)...")
try:
    ncnn_path = best_model.export(format='ncnn', imgsz=640, simplify=True)
    print(f"  NCNN exported to: {ncnn_path}")
except Exception as e:
    print(f"  NCNN export failed (optional, skipping): {e}")
    ncnn_path = None

print("\n\u2713 Step 8 (exports) complete: Models exported")

Exporting to ONNX format (primary)...
Ultralytics 8.4.51 🚀 Python-3.12.12 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/

PyTorch: starting from '/kaggle/working/runs/ardumedics_medium_pose/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 57, 8400) (50.8 MB)
requirements: Ultralytics requirements ['onnxslim>=0.1.71', 'onnxruntime-gpu'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /usr
Resolved 12 packages in 185ms
 Downloaded onnxruntime-gpu
Prepared 2 packages in 2.60s
Installed 2 packages in 16ms
 + onnxruntime-gpu==1.26.0
 + onnxslim==0.1.93

requirements: AutoUpdate success ✅ 3.3s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.20.1 opset 20...
ONNX: slimming with onnxslim 0.1.93...
ONNX: export success ✅ 7.9s

E0000 00:00:1779228303.120394      22 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779228303.183030      22 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779228303.712327      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779228303.712356      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779228303.712359      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779228303.712361      22 computation_placer.cc:177] computation placer already registered. Please check linka

requirements: Ultralytics requirements ['sng4onnx>=1.0.1', 'onnx_graphsurgeon>=0.3.26', 'ai-edge-litert>=1.2.0', 'onnx2tf>=1.26.3,<1.29.0'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /usr
Resolved 12 packages in 2.37s
 Downloaded ai-edge-litert
Prepared 5 packages in 248ms
Installed 5 packages in 7ms
 + ai-edge-litert==2.1.5
 + backports-strenum==1.3.1
 + onnx-graphsurgeon==0.6.1
 + onnx2tf==1.28.8
 + sng4onnx==2.0.1

requirements: AutoUpdate success ✅ 2.8s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


TensorFlow SavedModel: starting export with tensorflow 2.19.0...
Unzipping calibration_image_sample_data_20x128x128x3_float32.npy.zip to /kaggle/working/calibration_image_sample_data_20x128x128x3_float32.npy...: 100% ━━━━━━━━━━━━ 1/1 54.0files/s 0.0s
TensorFlow SavedModel: starting TFLite export with onnx2tf 1.28.8...


I0000 00:00:1779228330.428739      22 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 10257 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1779228330.433841      22 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1779228335.672567      22 cuda_dnn.cc:529] Loaded cuDNN version 91002


Saved artifact at '/kaggle/working/runs/ardumedics_medium_pose/weights/best_saved_model'. The following endpoints are available:

* Endpoint 'serving_default'
  inputs_0 (POSITIONAL_ONLY): TensorSpec(shape=(1, 640, 640, 3), dtype=tf.float32, name='images')
Output Type:
  TensorSpec(shape=(1, 57, 8400), dtype=tf.float32, name=None)
Captures:
  138634232059024: TensorSpec(shape=(4, 2), dtype=tf.int32, name=None)
  138634232057872: TensorSpec(shape=(3, 3, 3, 48), dtype=tf.float32, name=None)
  138634232058256: TensorSpec(shape=(48,), dtype=tf.float32, name=None)
  138634232063440: TensorSpec(shape=(4, 2), dtype=tf.int32, name=None)
  138634232063824: TensorSpec(shape=(3, 3, 48, 96), dtype=tf.float32, name=None)
  138634232061904: TensorSpec(shape=(96,), dtype=tf.float32, name=None)
  138634232064016: TensorSpec(shape=(1, 1, 96, 96), dtype=tf.float32, name=None)
  138634232064208: TensorSpec(shape=(96,), dtype=tf.float32, name=None)
  138634232065168: TensorSpec(shape=(4,), dtype=tf.int64,

I0000 00:00:1779228343.765515      22 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 2
I0000 00:00:1779228343.765706      22 single_machine.cc:374] Starting new session
I0000 00:00:1779228343.778927      22 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 10257 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1779228343.780363      22 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
W0000 00:00:1779228347.824587      22 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1779228347.824626      22 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
I0000 00:00:1779228351.280600      22 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 2
I0000 00:00:1779228351.280

TensorFlow SavedModel: export success ✅ 59.7s, saved as '/kaggle/working/runs/ardumedics_medium_pose/weights/best_saved_model' (253.5 MB)

TensorFlow Lite: starting export with tensorflow 2.19.0...
TensorFlow Lite: export success ✅ 0.0s, saved as '/kaggle/working/runs/ardumedics_medium_pose/weights/best_saved_model/best_float32.tflite' (101.3 MB)

Export complete (61.1s)
Results saved to /kaggle/working/runs/ardumedics_medium_pose/weights/best_saved_model/best_float32.tflite
Predict:         yolo predict task=pose model=/kaggle/working/runs/ardumedics_medium_pose/weights/best_saved_model/best_float32.tflite imgsz=640 
Validate:        yolo val task=pose model=/kaggle/working/runs/ardumedics_medium_pose/weights/best_saved_model/best_float32.tflite imgsz=640 data=/kaggle/working/datasets/ardumedics_pose.yaml  
Visualize:       https://netron.app
  TFLite exported to: /kaggle/working/runs/ardumedics_medium_pose/weights/best_saved_model/best_float32.tflite

Exporting to NCNN format (option

pnnxparam = /kaggle/working/runs/ardumedics_medium_pose/weights/best_ncnn_model/model.pnnx.param
pnnxbin = /kaggle/working/runs/ardumedics_medium_pose/weights/best_ncnn_model/model.pnnx.bin
pnnxpy = /kaggle/working/runs/ardumedics_medium_pose/weights/best_ncnn_model/model_pnnx.py
pnnxonnx = /kaggle/working/runs/ardumedics_medium_pose/weights/best_ncnn_model/model.pnnx.onnx
ncnnparam = /kaggle/working/runs/ardumedics_medium_pose/weights/best_ncnn_model/model.ncnn.param
ncnnbin = /kaggle/working/runs/ardumedics_medium_pose/weights/best_ncnn_model/model.ncnn.bin
ncnnpy = /kaggle/working/runs/ardumedics_medium_pose/weights/best_ncnn_model/model_ncnn.py
fp16 = 0
optlevel = 2
device = cpu
inputshape = [1,3,640,640]f32
inputshape2 = 
customop = 
moduleop = 
get inputshape from traced inputs
inputshape = [1,3,640,640]f32
############# pass_level0
inline module = torch.nn.modules.linear.Identity
inline module = ultralytics.nn.modules.block.Bottleneck
inline module = ultralytics.nn.modules.block

NCNN: export success ✅ 14.8s, saved as '/kaggle/working/runs/ardumedics_medium_pose/weights/best_ncnn_model' (101.1 MB)

Export complete (16.1s)
Results saved to /kaggle/working/runs/ardumedics_medium_pose/weights/best_ncnn_model
Predict:         yolo predict task=pose model=/kaggle/working/runs/ardumedics_medium_pose/weights/best_ncnn_model imgsz=640 
Validate:        yolo val task=pose model=/kaggle/working/runs/ardumedics_medium_pose/weights/best_ncnn_model imgsz=640 data=/kaggle/working/datasets/ardumedics_pose.yaml  
Visualize:       https://netron.app
  NCNN exported to: /kaggle/working/runs/ardumedics_medium_pose/weights/best_ncnn_model

✓ Step 8 (exports) complete: Models exported


In [12]:
# ============================================================
# STEP 8 (continued): Model size comparison & summary
# Notebook: 03 | Step: 8 of 8
# After this: Proceed to Notebook 04 (Knowledge Distillation)
# ============================================================

# ---- List exported model sizes for this notebook ----
print("=" * 70)
print("EXPORTED MODEL SIZES - YOLOv8m-Pose (Medium)")
print("=" * 70)

export_info = {}
pytorch_size_mb = os.path.getsize(best_model_path) / (1024 * 1024)
export_info['pytorch_size_mb'] = pytorch_size_mb
print(f"  {'PyTorch (best.pt)':30s}: {pytorch_size_mb:.2f} MB")

if onnx_path and os.path.exists(str(onnx_path)):
    onnx_size_mb = os.path.getsize(str(onnx_path)) / (1024 * 1024)
    export_info['onnx_size_mb'] = onnx_size_mb
    print(f"  {'ONNX':30s}: {onnx_size_mb:.2f} MB")

if tflite_path and os.path.exists(str(tflite_path)):
    tflite_size_mb = os.path.getsize(str(tflite_path)) / (1024 * 1024)
    export_info['tflite_size_mb'] = tflite_size_mb
    print(f"  {'TFLite':30s}: {tflite_size_mb:.2f} MB")

if ncnn_path and os.path.exists(str(ncnn_path)):
    ncnn_dir = str(ncnn_path)
    if os.path.isdir(ncnn_dir):
        ncnn_size = sum(os.path.getsize(os.path.join(ncnn_dir, f)) 
                        for f in os.listdir(ncnn_dir))
    else:
        ncnn_size = os.path.getsize(ncnn_dir)
    ncnn_size_mb = ncnn_size / (1024 * 1024)
    export_info['ncnn_size_mb'] = ncnn_size_mb
    print(f"  {'NCNN':30s}: {ncnn_size_mb:.2f} MB")

# ---- Cross-Notebook Model Size Comparison ----
print("\n" + "=" * 70)
print("CROSS-NOTEBOOK MODEL SIZE COMPARISON")
print("=" * 70)
print(f"  {'Notebook':35s} {'Model':15s} {'Params':>10s} {'PyTorch MB':>12s}")
print(f"  {'-'*35} {'-'*15} {'-'*10} {'-'*12}")
print(f"  {'NB01 - Nano (Edge Deploy)':35s} {'yolov8n-pose':15s} {'3.2M':>10s} {'~6.5':>12s}")
print(f"  {'NB02 - Small (Balanced)':35s} {'yolov8s-pose':15s} {'11.2M':>10s} {'~22.5':>12s}")
print(f"  {'NB03 - Medium (Upper Bound)':35s} {'yolov8m-pose':15s} {'26.4M':>10s} {pytorch_size_mb:>12.1f}")
print()
print("  NOTE: Medium model is NOT intended for Pi 5 real-time deployment.")
print("  It serves as: Teacher model (NB04), Ensemble member (NB06), Paper reference.")

# ---- Update metrics JSON with export info ----
metrics_03['exports'] = export_info
with open(metrics_path, 'w') as f:
    json.dump(metrics_03, f, indent=2)

print(f"\nMetrics (with exports) saved to: {metrics_path}")

EXPORTED MODEL SIZES - YOLOv8m-Pose (Medium)
  PyTorch (best.pt)             : 50.79 MB
  ONNX                          : 101.36 MB
  TFLite                        : 101.29 MB
  NCNN                          : 101.08 MB

CROSS-NOTEBOOK MODEL SIZE COMPARISON
  Notebook                            Model               Params   PyTorch MB
  ----------------------------------- --------------- ---------- ------------
  NB01 - Nano (Edge Deploy)           yolov8n-pose          3.2M         ~6.5
  NB02 - Small (Balanced)             yolov8s-pose         11.2M        ~22.5
  NB03 - Medium (Upper Bound)         yolov8m-pose         26.4M         50.8

  NOTE: Medium model is NOT intended for Pi 5 real-time deployment.
  It serves as: Teacher model (NB04), Ensemble member (NB06), Paper reference.

Metrics (with exports) saved to: /kaggle/working/metrics_notebook03_medium.json


---
## Step 9: Save Outputs for Next Notebooks

Pack all outputs (trained model, metrics JSON, exports) into a directory
that can be saved as a **Kaggle Dataset** for use by Notebooks 05 and 06.

**Instructions**: After this notebook completes, create a new Kaggle Dataset
from the output. Name it `ardumedics-nb03-medium-outputs`.

In [13]:
# ============================================================
# STEP 9: Save outputs for next notebooks
# Pack everything into /kaggle/working/nb03_outputs/ for Kaggle Dataset
# ============================================================

OUTPUT_PACK_DIR = '/kaggle/working/nb03_outputs'
os.makedirs(OUTPUT_PACK_DIR, exist_ok=True)

import shutil

# Copy best model weights
best_pt_src = '/kaggle/working/runs/ardumedics_medium_pose/weights/best.pt'
if os.path.exists(best_pt_src):
    shutil.copy2(best_pt_src, f'{OUTPUT_PACK_DIR}/best_medium.pt')
    print(f'  Copied best_medium.pt ({os.path.getsize(f"{OUTPUT_PACK_DIR}/best_medium.pt")/1e6:.1f} MB)')

# Copy metrics JSON
metrics_src = '/kaggle/working/metrics_notebook03_medium.json'
if os.path.exists(metrics_src):
    shutil.copy2(metrics_src, f'{OUTPUT_PACK_DIR}/metrics_notebook03_medium.json')
    print('  Copied metrics_notebook03_medium.json')

# Copy exported models (if any)
export_dir = '/kaggle/working/exports'
if os.path.exists(export_dir):
    for item in os.listdir(export_dir):
        src = os.path.join(export_dir, item)
        dst = os.path.join(OUTPUT_PACK_DIR, item)
        if os.path.isdir(src):
            if not os.path.exists(dst):
                shutil.copytree(src, dst)
        else:
            shutil.copy2(src, dst)
    print('  Copied export files')

print(f'\nAll outputs packed to: {OUTPUT_PACK_DIR}')
print('\n>>> IMPORTANT: Save this output as a Kaggle Dataset! <<<')
print('    1. Click "Save Version" to run the notebook')
print('    2. After completion, go to Output tab')
print('    3. Create a new Dataset from the nb03_outputs/ folder')
print('    4. Name it: ardumedics-nb03-medium-outputs')
print('    5. Add this dataset as input to NB05 and NB06')
print('\n✓ Step 9 complete: Outputs packed for next notebooks')

  Copied best_medium.pt (53.3 MB)
  Copied metrics_notebook03_medium.json
  Copied export files

All outputs packed to: /kaggle/working/nb03_outputs

>>> IMPORTANT: Save this output as a Kaggle Dataset! <<<
    1. Click "Save Version" to run the notebook
    2. After completion, go to Output tab
    3. Create a new Dataset from the nb03_outputs/ folder
    4. Name it: ardumedics-nb03-medium-outputs
    5. Add this dataset as input to NB05 and NB06

✓ Step 9 complete: Outputs packed for next notebooks


In [14]:
# ============================================================
# STEP 8 (continued): Final summary and next steps
# Notebook: 03 | Step: 8 of 8
# After this: Proceed to Notebook 04 (Knowledge Distillation)
# ============================================================

print("\n" + "=" * 70)
print("NOTEBOOK 03 COMPLETE: YOLOv8m-Pose Fall Detection Training")
print("=" * 70)
print()
print("SUMMARY:")
print(f"  Model:           YOLOv8m-Pose (Medium - 26.4M params)")
print(f"  Training epochs: {TRAIN_CONFIG['epochs']}")
print(f"  Best model path: {best_model_path}")
print(f"  Metrics saved:   {metrics_path}")
print()
print("KEY RESULTS:")
print(f"  Val mAP@50 (Box):   {val_results.box.map50:.4f}")
print(f"  Val mAP@50 (Pose):  {val_results.pose.map50:.4f}")
print(f"  Test mAP@50 (Box):  {test_results.box.map50:.4f}")
print(f"  Test mAP@50 (Pose): {test_results.pose.map50:.4f}")
print()
print("MODEL ROLE IN PROJECT:")
print("  1. Teacher model for Knowledge Distillation (Notebook 04)")
print("  2. Ensemble member for boosted accuracy (Notebook 06)")
print("  3. Upper-bound accuracy reference for paper")
print()
print("NEXT STEPS:")
print("  Notebook 04: YOLOv8n-Pose Knowledge Distillation")
print("  - Use this medium model as teacher to distill knowledge into nano model")
print("  - Goal: Nano model accuracy approaching medium model")
print()
print("FILES TO SAVE (Kaggle Output):")
print(f"  {best_model_path}")
print(f"  {metrics_path}")
print(f"  {onnx_path}")
if tflite_path:
    print(f"  {tflite_path}")
if ncnn_path:
    print(f"  {ncnn_path}")
print("  /kaggle/working/runs/ardumedics_medium_pose/")
print()
print("\u2713 Notebook 03 complete!")


NOTEBOOK 03 COMPLETE: YOLOv8m-Pose Fall Detection Training

SUMMARY:
  Model:           YOLOv8m-Pose (Medium - 26.4M params)
  Training epochs: 80
  Best model path: /kaggle/working/runs/ardumedics_medium_pose/weights/best.pt
  Metrics saved:   /kaggle/working/metrics_notebook03_medium.json

KEY RESULTS:
  Val mAP@50 (Box):   0.7088
  Val mAP@50 (Pose):  0.1975
  Test mAP@50 (Box):  0.7488
  Test mAP@50 (Pose): 0.2591

MODEL ROLE IN PROJECT:
  1. Teacher model for Knowledge Distillation (Notebook 04)
  2. Ensemble member for boosted accuracy (Notebook 06)
  3. Upper-bound accuracy reference for paper

NEXT STEPS:
  Notebook 04: YOLOv8n-Pose Knowledge Distillation
  - Use this medium model as teacher to distill knowledge into nano model
  - Goal: Nano model accuracy approaching medium model

FILES TO SAVE (Kaggle Output):
  /kaggle/working/runs/ardumedics_medium_pose/weights/best.pt
  /kaggle/working/metrics_notebook03_medium.json
  /kaggle/working/runs/ardumedics_medium_pose/weights/b